# 02 — EDA: genomic (synthetic)

Compare **baseline** synthetic genotype weights with the optional **West Africa rs334 MAF** tilt (`west_africa_rs334_maf`). The latter is a *benchmark prior only* — not a substitute for real genotypes.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/koneke55/Mmvlm4SCD/blob/main/notebooks/02-eda-genomic.ipynb)

**Google Colab + GPU:** [Unsloth](https://unsloth.ai) documents a practical [Google Colab workflow](https://docs.unsloth.ai/get-started/install/google-colab) (free **T4** GPU tier, Runtime menu, run cells in order). Use it as the reference for attaching hardware acceleration.

**Note:** This repo does **not** depend on the `unsloth` pip package—only standard PyTorch + `pip install -e .`; the Unsloth guide covers Colab compute ergonomics.

**Local:** run `pip install -e .` from the repo root. **Colab:** run the environment cell below (clone under `/content` when needed).


## 1. Environment setup (Colab or local)

- **Colab:** optional `MMVLM_REPO_URL` for your fork; defaults to upstream.
- Installs this package editable (`pip install -e .`).


In [ ]:
import os
import subprocess
import sys


def _in_colab() -> bool:
    try:
        import google.colab  # noqa: F401
        return True
    except ImportError:
        return False


def _find_repo_root(start: str) -> str:
    cur = os.path.abspath(start)
    for _ in range(8):
        if os.path.isdir(os.path.join(cur, "src", "mmvlm4scd")):
            return cur
        parent = os.path.dirname(cur)
        if parent == cur:
            break
        cur = parent
    raise RuntimeError(
        "Could not find mmvlm4scd package root (missing src/mmvlm4scd). "
        "Open the notebook from the repo or run the Colab clone cell."
    )


if _in_colab():
    REPO_URL = os.environ.get(
        "MMVLM_REPO_URL",
        "https://github.com/koneke55/Mmvlm4SCD.git",
    )
    DEST = "/content/Mmvlm4SCD"
    if not os.path.isdir(os.path.join(DEST, "src", "mmvlm4scd")):
        subprocess.check_call(
            ["git", "clone", "--depth", "1", REPO_URL, DEST],
            stdout=subprocess.DEVNULL,
        )
    os.chdir(DEST)
    ROOT = DEST
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])
else:
    ROOT = _find_repo_root(os.getcwd())
    os.chdir(ROOT)
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "-e", "."])

sys.path.insert(0, os.path.join(ROOT, "src"))
print("Repo root:", ROOT)


## 2. Accelerator check

Mirrors the GPU verification pattern recommended alongside [Unsloth's Colab instructions](https://docs.unsloth.ai/get-started/install/google-colab).


In [ ]:
import torch

print("torch:", torch.__version__)
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))
else:
    print("CPU-only runtime — for GPU follow Unsloth's Colab guide (Runtime → Change runtime type).")


## 3. Imports


In [ ]:
import matplotlib.pyplot as plt
import numpy as np

from mmvlm4scd.data import generate_synthetic_cohort
from mmvlm4scd.data.synthetic import SCDSyntheticConfig


In [ ]:
def genotype_props(cfg):
    c = generate_synthetic_cohort(cfg)
    g = c["clinical"]["genotype"].values
    labs, cnt = np.unique(g, return_counts=True)
    return dict(zip(labs, cnt / cnt.sum()))

base = genotype_props(SCDSyntheticConfig(n_patients=8000, seed=0))
wa = genotype_props(SCDSyntheticConfig(n_patients=8000, seed=0, west_africa_rs334_maf=0.13))

labels = sorted(set(base) | set(wa))
x = np.arange(len(labels))
w = 0.35
plt.bar(x - w / 2, [base[k] for k in labels], width=w, label="baseline")
plt.bar(x + w / 2, [wa[k] for k in labels], width=w, label="west_africa_rs334_maf=0.13")
plt.xticks(x, labels, rotation=30, ha="right")
plt.ylabel("fraction")
plt.title("Synthetic genotype mix: baseline vs West Africa MAF tilt")
plt.legend()
plt.tight_layout()
plt.show()


## Variant-indicator block (first 16 dims)

Column 0 is aligned with higher genotype severity in the simulator.


In [ ]:
cohort = generate_synthetic_cohort(SCDSyntheticConfig(n_patients=2000, seed=1))
g = cohort["genomic"]
print("genomic shape (N, 32):", g.shape)
print("mean allele-indicator dims 0–15:", g[:, :16].mean(axis=0)[:8])
